# C3 exp03: does a bigger model fix the letter bias?

exp01's setting (zero-shot, k = 0, definitions on) with three larger Qwen2.5 Instruct models, each also calibrated (as exp02):
- **1.5B** (fp16) on GPU 0 and **3B** (fp16) on GPU 1, at the same time;
- then **7B** in 4-bit on GPU 0 (7B in fp16 needs ~15 GB, more than a T4 has).

Note: `tasks/C3-prompting-baseline/experiments/exp03_model-size/NOTE.md`. Temporary trials (`scripts/trial_prompting.py`); not reportable (PROTOCOL §4.1).

Before running, in the notebook settings: **GPU T4 x2**, **Internet on**, the `smi-v3-meetings` dataset attached.

Expected time: roughly 30–40 min for 1.5B and 3B together, then ~1–2 h for 7B (estimates; the logs show the real speed).

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
`bitsandbytes` is needed for the 4-bit 7B model. Do **not** install `requirements.txt` here.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C3/prompting-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 bitsandbytes

## 3. Data folder

In [ ]:
import glob, json, os, subprocess, time

hits = glob.glob("/kaggle/input/**/source.json", recursive=True)
assert len(hits) == 1, "expected exactly one source.json under /kaggle/input : is the dataset attached?"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "|", json.load(open(hits[0]))["data_version"])
os.makedirs("/kaggle/working/logs", exist_ok=True)

## 4. Helpers: start a run on one GPU, and wait for runs while showing their progress

In [ ]:
def launch(gpu, run_name, model, extra=()):
    """start trial_prompting.py in the background on one GPU ; its output goes to logs/<run_name>.log"""
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src")
    cmd = ["python", "scripts/trial_prompting.py", "--data-root", DATA_ROOT, "--out-dir", "/kaggle/working/runs",
           "--run-name", run_name, "--model", model, "--batch-size", "16", *extra]
    log = open(f"/kaggle/working/logs/{run_name}.log", "w")
    return subprocess.Popen(cmd, cwd="/kaggle/working/repo", env=env, stdout=log, stderr=subprocess.STDOUT)

def last_line(run_name):
    lines = [l for l in open(f"/kaggle/working/logs/{run_name}.log").read().splitlines() if l.strip()]
    return lines[-1] if lines else "(starting)"

def wait(processes):
    """print each run's latest progress line every 2 minutes until all have finished"""
    while any(p.poll() is None for p in processes.values()):
        time.sleep(120)
        for name in processes:
            print(f"{name:20} {last_line(name)[-90:]}")
        print()
    for name, p in processes.items():
        print(name, "finished OK" if p.returncode == 0 else f"FAILED (code {p.returncode}) : see logs/{name}.log")

## 5. 1.5B on GPU 0 and 3B on GPU 1, in parallel

In [ ]:
wait({
    "c3_exp03_1.5b": launch(0, "c3_exp03_1.5b", "Qwen/Qwen2.5-1.5B-Instruct"),
    "c3_exp03_3b": launch(1, "c3_exp03_3b", "Qwen/Qwen2.5-3B-Instruct"),
})

## 6. 7B in 4-bit on GPU 0
If the session stops, run this cell again: the run continues where it stopped.

In [ ]:
wait({"c3_exp03_7b": launch(0, "c3_exp03_7b", "Qwen/Qwen2.5-7B-Instruct", ["--load-in-4bit"])})

## 7. Calibrate each run (as exp02)

In [ ]:
!PYTHONPATH=src python scripts/calibrate_prompting.py --data-root {DATA_ROOT} --run /kaggle/working/runs/c3_exp03_1.5b
!PYTHONPATH=src python scripts/calibrate_prompting.py --data-root {DATA_ROOT} --run /kaggle/working/runs/c3_exp03_3b
!PYTHONPATH=src python scripts/calibrate_prompting.py --data-root {DATA_ROOT} --run /kaggle/working/runs/c3_exp03_7b

## 8. Results

In [ ]:
print(f"{'run':24} {'macro-F1':>8} {'other F1':>8} {'mass':>5}  predicted labels")
for name in ["c3_exp03_1.5b", "c3_exp03_1.5b_cc", "c3_exp03_3b", "c3_exp03_3b_cc", "c3_exp03_7b", "c3_exp03_7b_cc"]:
    path = f"/kaggle/working/runs/{name}"
    if not os.path.exists(f"{path}/metrics.json"):
        print(f"{name:24} (missing)"); continue
    m = json.load(open(f"{path}/metrics.json"))
    predicted = {}
    for line in open(f"{path}/predictions.jsonl"):
        label = json.loads(line)["label"]
        predicted[label] = predicted.get(label, 0) + 1
    mass = m.get("prompting", {}).get("mean_letter_mass")
    mass = f"{mass:5.3f}" if mass is not None else "  -  "
    print(f"{name:24} {m['primary']['macro_f1_6']:8.4f} {m['per_class']['other']['f1']:8.3f} {mass}  {predicted}")
    print(" " * 24, "per-class F1:", {k: round(v["f1"], 3) for k, v in m["per_class"].items()})

## 9. Download the outputs
Right-hand panel: **Output** → `/kaggle/working` → `c3_exp03_runs.zip` → **⋮** → **Download**.

In [ ]:
%cd /kaggle/working
!zip -qr c3_exp03_runs.zip runs logs
!ls -la c3_exp03_runs.zip